#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

#Reading From Bronze Layer

In [0]:
df=spark.table("workspace.bronze.crm_cust_info")

#Data Transformation

#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

In [0]:
df=(
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status"))=='M','Married')
        .when(F.upper(F.col("cst_marital_status"))=='S','Single')
        .otherwise('n/a')
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr"))=='M','Male')
        .when(F.upper(F.col("cst_gndr"))=='F','Female')
        .otherwise('n/a')
    )

)


#Renaming columns

In [0]:
rename_cols={
    "cst_id":"customer_id",
    "cst_key":"customer_number",
    "cst_firstname":"first_name",
    "cst_lastname":"last_name",
    "cst_marital_status":"marital_status",
    "cst_gndr":"gender",
    "cst_create_date":"created_date"
    }
   


for old_name,new_name in rename_cols.items():
    df=df.withColumnRenamed(old_name,new_name)


#Write to Silver Layer

In [0]:
(
df.write
.mode("overwrite")
.format('delta')
.saveAsTable("silver.crm_customers")
)